# Nile-Chat 12B v2 — Merge, Push to Hugging Face, and Serve

Merges the v2 LoRA adapter (trained in `nile_chat_finetune_v2_colab.ipynb`) into the
base model, pushes the merged model to a **new** Hugging Face repo
(`mennaharmas/raylab-nilechat-12b-v2` — separate from the v1 repo `mennaharmas/raylab-nilechat-12b`, so the
currently-deployed v1 model is never overwritten while v2 is being validated), and
serves it with vLLM for real testing.

**Security note on the v1 notebook this is adapted from**: its serve cell had a real
Hugging Face token hardcoded in plaintext (`os.environ["HF_TOKEN"] = "hf_..."`).
It was not committed to git, but it's a live secret sitting in a local file — rotate
that token on huggingface.co/settings/tokens as a precaution, and this notebook
never hardcodes one — it reads from Colab's secret store (`userdata`) the same way
the training notebook already does for the Hub login.


### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/gdrive')


Mounted at /gdrive


## Stage 12 — Merge LoRA adapter into base model

In [1]:
# Same install as the training notebook -- a fresh runtime needs it again.
!pip uninstall -y torch torchvision torchaudio vllm
!pip install -qU uv
!uv pip install --system vllm --torch-backend=auto
!pip uninstall -y torchao
!pip install -qU "transformers>=4.55.0,<=5.8.0,!=4.57.0,!=5.6.0" "datasets>=2.16.0,<=4.0.0" "accelerate>=1.3.0,<=1.15.0" "peft>=0.18.0,<=0.20.0" "trl>=0.18.0,<=0.24.0"
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e . --no-deps


Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 123.4 MB/s eta 0:00:00
Using Python 3.13.15 environment at: /usr
Resolved 196 packages in 2.80s
Prepared 101 packages in 46.96s
Uninstalled 14 packages in 163ms
Installed 101 packages in 404ms
 + agent-detector==1.1.0
 + anthropic==1.2.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + blake3==1.0.9
 + cbor2==6.1.4
 + compressed-tensors==0.17.0
 - cuda-bindings==12.9.7
 + cuda-bindings==13.3.1
 - cuda-core==0.3.2
 + cuda-core==1.0.1
 - cuda-python==12.9.7
 + cuda-python==13.3.1
 + cuda-tile==1.5.0
 - cuda-toolkit==12.8.1
 + cud

In [2]:
import torch

torch_version = torch.__version__.split("+")[0]
torch_cuda = torch.version.cuda
cuda_tag = "cu" + torch_cuda.replace(".", "")

print(f"Detected torch=={torch_version} built for CUDA {torch_cuda} -> installing matched "
      f"torchvision from index {cuda_tag}, leaving torchaudio uninstalled")

!pip install -q "torch=={torch_version}" torchvision --index-url https://download.pytorch.org/whl/{cuda_tag}
!pip uninstall -y -q torchaudio

import subprocess
check = subprocess.run(
    ["python", "-c", "import torch, torchvision; "
     "print('torch:', torch.__version__, torch.version.cuda); "
     "print('torchvision:', torchvision.__version__)"],
    capture_output=True, text=True,
)
print(check.stdout.strip())
assert check.returncode == 0, f"torch/torchvision import failing:\n{check.stderr}"
print("torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.")


Detected torch==2.13.0 built for CUDA 13.2 -> installing matched torchvision from index cu132, leaving torchaudio uninstalled
torch: 2.13.0+cu132 13.2
torchvision: 0.28.0+cu132
torch/torchvision aligned and importable. torchaudio intentionally left uninstalled.


In [3]:
import os
from google.colab import userdata

# Token from Colab's secret store -- never hardcoded (see this notebook's intro cell
# re: the v1 notebook's leaked plaintext token). Only needed because the repo above
# is private.
os.environ["HF_TOKEN"] = userdata.get('huggingface')

!nohup vllm serve "mennaharmas/raylab-nilechat-12b-v2" \
    --dtype bfloat16 \
    --max-model-len 8192 \
    --gpu-memory-utilization 0.85 \
    --port 8001 \
    --served-model-name raylab-nilechat-finetuned \
    > vllm.log 2>&1 &


In [4]:
import time

ready = False
for attempt in range(60):  # up to 10 minutes
    time.sleep(10)
    log = open("vllm.log").read() if __import__("os").path.exists("vllm.log") else ""
    if "Uvicorn running" in log or "Application startup complete" in log:
        ready = True
        break
    if "Traceback" in log and "ERROR" in log:
        print("vLLM logged an error while loading -- check the tail below.")
        break
    print(f"[{(attempt + 1) * 10}s] still loading...")

!tail -n 60 vllm.log
print("\n--- server ready:", ready, "---\n")

!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"raylab-nilechat-finetuned","messages":[{"role":"user","content":"hi"}],"max_tokens":16}'


[10s] still loading...
[20s] still loading...
[30s] still loading...
[40s] still loading...
[50s] still loading...
[60s] still loading...
[70s] still loading...
[80s] still loading...
[90s] still loading...
[100s] still loading...
[110s] still loading...
[120s] still loading...
[130s] still loading...
[140s] still loading...
[150s] still loading...
[160s] still loading...
[170s] still loading...
[180s] still loading...
[190s] still loading...
[200s] still loading...
[210s] still loading...
[220s] still loading...
[230s] still loading...
[240s] still loading...
[250s] still loading...
[260s] still loading...
[270s] still loading...
[280s] still loading...
[290s] still loading...
[300s] still loading...
[310s] still loading...
[320s] still loading...
[330s] still loading...
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:06<00:00,  1.38s/it]
(EngineCore pid=5806) 
(EngineCore pid=5806) INFO 09-01 16:38:06 [default_loader.py:430] Loading weights took 6.97 seconds
(EngineCo

If the curl call above didn't return a real completion, stop and fix it before opening a tunnel.


In [5]:
import os, re, time

if not os.path.exists("cloudflared-linux-amd64"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

assert os.path.exists("cloudflared-linux-amd64") and os.path.getsize("cloudflared-linux-amd64") > 0, \
    "cloudflared download failed -- re-run this cell, or check Colab's network connectivity"

!chmod +x cloudflared-linux-amd64
!nohup ./cloudflared-linux-amd64 tunnel --url http://localhost:8001 > cloudflared.log 2>&1 &

tunnel_url = None
for _ in range(30):
    time.sleep(2)
    log = open("cloudflared.log").read() if os.path.exists("cloudflared.log") else ""
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log)
    if match:
        tunnel_url = match.group(0)
        break

assert tunnel_url, "Tunnel URL not found after 60s -- check cloudflared.log for errors and re-run this cell"
print(f"Tunnel URL: {tunnel_url}")
print("Set in your local src/.env:")
print(f"  GENERATION_BASE_URL={tunnel_url}")
print("  GENERATION_MODEL_NAME=raylab-nilechat-finetuned")


Tunnel URL: https://adelaide-graduation-liability-caribbean.trycloudflare.com
Set in your local src/.env:
  GENERATION_BASE_URL=https://adelaide-graduation-liability-caribbean.trycloudflare.com
  GENERATION_MODEL_NAME=raylab-nilechat-finetuned


In [9]:
!curl -s http://localhost:8001/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{"model":"raylab-nilechat-finetuned","messages":[{"role":"user","content":"إزيك؟"}],"max_tokens":32}'


{"id":"chatcmpl-9f87d70b2c27ce28","object":"chat.completion","created":1788284311,"model":"raylab-nilechat-finetuned","choices":[{"index":0,"message":{"role":"assistant","content":"```json\n{}\n```\n\nالحمد لله يا فندم، وإنت عامل إيه؟ أقدر أساعدك في حاجة النهاردة؟","refusal":null,"annotations":null,"audio":null,"function_call":null,"reasoning":null},"logprobs":null,"finish_reason":"length","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.28.0-ffd20a0f","usage":{"prompt_tokens":13,"total_tokens":45,"completion_tokens":32,"prompt_tokens_details":null,"completion_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null,"ec_transfer_params":null,"metrics":null}